# imports

In [ ]:
import os
from glob import glob 
import subprocess as sp 
import boto3
from botocore import UNSIGNED
from botocore.client import Config
import pandas as pd 
import numpy as np 
import multiprocessing as mp 

# functions to download files in a directory (full or partial [e.g., hourly])

In [1]:
def download_s3_folder(bucket_name, obj_name, dates, local_dir):
    for d in dates:
        dstr = d.strftime("%Y%m%d")
        outdir = f'{local_dir}/{obj_name}/{dstr}' 
        os.makedirs(outdir, exist_ok=True)
        prefix = f"{obj_name}/{dstr}/"
                
        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
            for obj in page.get('Contents', []):
                key = obj['Key']
                filename = os.path.basename(key)
                if not filename:  # skip directories
                    continue
                local_path = os.path.join(outdir, filename)
                s3.download_file(bucket_name, key, local_path)
        print(f"✅ All files in {prefix} downloaded successfully.")
        
        # extra 
        cmd = f'gunzip {outdir}/*.gz'
        sp.run(cmd, shell=True)   
        
        
def download_s3_subfolder(bucket_name, obj_name, dates, local_dir):
    for d in dates:
        dstr = d.strftime("%Y%m%d")
        outdir = f'{local_dir}/{obj_name}/{dstr}' 
        os.makedirs(outdir, exist_ok=True)
        prefix = f"{obj_name}/{dstr}/"
                
        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
            for obj in page.get('Contents', []):
                key = obj['Key']
                filename = os.path.basename(key)
                if not filename:  # skip directories
                    continue
                timestamp = filename.split('-')[-1].split('.')[0]
                if not timestamp[2:4] == '00':  # MM must be 00 for hourly files 
                    continue
                local_path = os.path.join(outdir, filename)
                s3.download_file(bucket_name, key, local_path)
        print(f"✅ All hourly files in {prefix} downloaded successfully.")
        
        # extra 
        cmd = f'gunzip {outdir}/*.gz'
        sp.run(cmd, shell=True) 

# example to download hourly precipitation (full dir) and 3d reflectivity (partial dir)

In [ ]:
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
bucket_name = "noaa-mrms-pds"
paginator = s3.get_paginator('list_objects_v2') 

dates = pd.date_range('20250501', '20250531', freq='1d')
local_dir = "/global/cfs/cdirs/m1657/meng/obs/mrms" 

# precipitation
obj_name = 'CONUS/MultiSensor_QPE_01H_Pass2_00.00'
download_s3_folder(bucket_name, obj_name, dates, local_dir)

# 3d reflectivity
heights = ['00.50', '00.75', '01.00', '01.25', '01.50', '01.75', '02.00', '02.25', '02.50', '02.75', 
           '03.00', '03.50', '04.00', '04.50', '05.00', '05.50', '06.00', '06.50', '07.00', '07.50',
           '08.00', '08.50', '09.00', '10.00', '11.00', '12.00', '13.00', '14.00', '15.00', '16.00',
           '17.00', '18.00', '19.00']
for z in heights:
    obj_name = f'CONUS/MergedReflectivityQC_{z}'
    download_s3_subfolder(bucket_name, obj_name, dates, local_dir)

# parallel downloading (recommended)

In [ ]:
def download_s3_subfolder_mp(obj_name):
    for d in dates:
        dstr = d.strftime("%Y%m%d")
        outdir = f'{local_dir}/{obj_name}/{dstr}' 
        os.makedirs(outdir, exist_ok=True)
        prefix = f"{obj_name}/{dstr}/"
                
        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
            for obj in page.get('Contents', []):
                key = obj['Key']
                filename = os.path.basename(key)
                if not filename:  # skip directories
                    continue
                timestamp = filename.split('-')[-1].split('.')[0]
                if not timestamp[2:4] == '00':  # MM must be 00
                    continue
                local_path = os.path.join(outdir, filename)
                s3.download_file(bucket_name, key, local_path)
        print(f"✅ All hourly files in {obj_name}/{prefix} downloaded successfully.")
        
        # extra 
        cmd = f'gunzip {outdir}/*.gz'
        sp.run(cmd, shell=True)  
        

heights = ['00.50', '00.75', '01.00', '01.25', '01.50', '01.75', '02.00', '02.25', '02.50', '02.75', 
           '03.00', '03.50', '04.00', '04.50', '05.00', '05.50', '06.00', '06.50', '07.00', '07.50',
           '08.00', '08.50', '09.00', '10.00', '11.00', '12.00', '13.00', '14.00', '15.00', '16.00',
           '17.00', '18.00', '19.00']
obj_names = [f'CONUS/MergedReflectivityQC_{z}' for z in heights]

num_workers = len(heights)
pool = mp.Pool(num_workers)
list(pool.imap(download_s3_subfolder_mp, obj_names))
pool.close()
pool.join() 

# dirty fix for 20250520
There are no hh:00:ss 3d reflectivity files, so use hh:02:ss 

```diff
- if not timestamp[2:4] == '00':  # MM must be 00 for hourly files 
+ if not timestamp[0:4] == '0002':  # Use hh:02:ss files for 20250520 workaround
```


# coarsen and save the data in netcdf format for PyFlexTrkr

In [ ]:
# SEUS domain
res = 0.0325  # nlat=234, nlon=438
lat_s, lat_n, lon_w, lon_e = [30.7, 38.3, 266.5, 280.7]
lat_b = np.linspace(lat_s, lat_n, 234+1)
lon_b = np.linspace(lon_w, lon_e, 438+1)
lats = 0.5 * (lat_b[1:] + lat_b[:-1])
lons = 0.5 * (lon_b[1:] + lon_b[:-1])
gridout = {'lon': lons, 'lat': lats, 'lon_b': lon_b, 'lat_b': lat_b}

do_regridder = False  
weights = '/global/cfs/cdirs/e3sm/meng/climo/maps/regridder_mrms_to_rrm_seus_3.25km_conservative.nc' 

heights = ['00.50', '00.75', '01.00', '01.25', '01.50', '01.75', '02.00', '02.25', '02.50', '02.75', 
           '03.00', '03.50', '04.00', '04.50', '05.00', '05.50', '06.00', '06.50', '07.00', '07.50',
           '08.00', '08.50', '09.00', '10.00', '11.00', '12.00', '13.00', '14.00', '15.00', '16.00',
           '17.00', '18.00', '19.00']
heights = np.array(heights).astype(float)

times = pd.date_range("2025-05-01", "2025-05-31", freq="1d")
indir = '/global/cfs/cdirs/m1657/meng/obs/mrms/CONUS'
outdir = '/pscratch/sd/m/meng/seus_output/mrms_remapped/for_idcell'

def remap_mrms(date):
    tid = date.strftime("%Y%m%d") 

    precip_list = []
    dbz_3d_list = []

    nt = 24  # hours in a day
    coord_t = pd.date_range(tid, periods=nt, freq='h')

    first_file = True 
    for i in range(nt):  # hourly data
        hour = f'{i:02d}'        
        ## precipitation 
        mrms = f'{indir}/MultiSensor_QPE_01H_Pass2_00.00/{tid}/MRMS_MultiSensor_QPE_01H_Pass2_00.00_{tid}-{hour}0000.grib2'
        print(f'Processing MRMS file {os.path.basename(mrms)}...\n') 
        mrms_ds = xr.open_dataset(mrms).rename({'unknown': 'precip'})
        
        if first_file:
            if do_regridder: 
                regridder = xe.Regridder(mrms_ds, xr.Dataset(gridout), 'conservative')
                regridder.to_netcdf(weights) 
            else:
                regridder = xe.Regridder(mrms_ds, xr.Dataset(gridout), 'conservative', weights=weights)      

        mrms_ds['precip'].values = xr.where(mrms_ds['precip'] < 0, np.nan, mrms_ds['precip'].values)  # -1 missing -3 no coverage
        precip_out = regridder(mrms_ds['precip'], skipna=True, na_thres=1)
        precip_list.append(precip_out)
        
        ## 3d reflectivity 
        dbz_3d = []
        for h in heights:
            ref = glob(f'{indir}/MergedReflectivityQC_{h:05.2f}/{tid}/MRMS_MergedReflectivityQC_{h:05.2f}_{tid}-{hour}00??.grib2')[0]
            print(f'Processing MRMS file {os.path.basename(ref)}...\n')
            ref_ds = xr.open_dataset(ref).rename({'unknown': 'reflectivity'}) 
            ref_ds['reflectivity'].values = xr.where(ref_ds['reflectivity'] <= -99, np.nan, ref_ds['reflectivity'].values)
            dbz_out = regridder(ref_ds['reflectivity'], skipna=True, na_thres=1) 
            dbz_3d.append(dbz_out)         
        dbz_3d = xr.concat(dbz_3d, dim='heightAboveSea') 
        dbz_3d_list.append(dbz_3d) 
        first_file = False  

    out_precip = xr.concat(precip_list, dim='time')   
    out_precip['time'] = coord_t  
    out_precip = out_precip.drop_vars([v for v in out_precip.coords if v not in ['time', 'lon', 'lat']]) 
    out_precip.attrs['units'] = 'mm (1-hr accum)'
    out_precip.attrs['long_name'] = 'MultiSensor_QPE_01H_Pass2_00.00'
        
    out_dbz = xr.concat(dbz_3d_list, dim='time')
    out_dbz['time'] = coord_t
    out_dbz['heightAboveSea'] = heights 
    out_dbz = out_dbz.drop_vars([v for v in out_dbz.coords if v not in ['time', 'lon', 'lat', 'heightAboveSea']])
    out_dbz.attrs['units'] = 'dBZ'
    out_dbz.attrs['long_name'] = 'MergedReflectivityQC' 

    dsout = xr.Dataset({
        'precip': out_precip,
        'reflectivity': out_dbz,
    })

    comp = {'_FillValue': None}
    encoding = {var: comp for var in dsout.variables}
    # for comperssion
    encoding.update({
        'precip': {'zlib': True, 'complevel': 5, 'shuffle': True, 'chunksizes': (nt, 234, 438)},
        'reflectivity': {'zlib': True, 'complevel': 5, 'shuffle': True, 'chunksizes': (nt, len(heights), 234, 438)},
    })
    encoding["time"] = {
        "dtype": "float32",
        "units": "hours since 1970-01-01 00:00:00",
        "calendar": "standard",
        "_FillValue": None
    }
    dsout.to_netcdf(f'{outdir}/mrms_rrm_seus_3.25km_{tid}.nc', encoding=encoding, unlimited_dims='time') 
    print(f'{tid} completed!')


num_workers = 8  #mp.cpu_count()//2 
pool = mp.Pool(num_workers)
list(pool.imap(remap_mrms, times))
pool.close()
pool.join() 